In [ ]:
# 1. IMPORT DES LIBRAIRIES
import pandas as pd
import numpy as np
from google.cloud import bigquery
import seaborn as sns
import matplotlib.pyplot as plt

# 2. CHARGEMENT DE LA DONNÉE (Source unique : mart_analysis_full)
client = bigquery.Client()
query = "SELECT * FROM `ton_projet.dataset_marts.mart_analysis_full`"
df = client.query(query).to_dataframe()

# 3. AUDIT DE LA QUALITÉ (Data Health Check)
def audit_data(df):
    report = {
        "Nombre total de lignes": len(df),
        "Valeurs manquantes (%)": df.isnull().mean() * 100,
        "Doublons": df.duplicated().sum()
    }
    return report

print("--- RAPPORT D'AUDIT QUALITÉ ---")
print(pd.DataFrame(audit_data(df)))

# 4. DÉTECTION DES ANOMALIES (Outliers & Aberrations)
# Identification des prix aberrants (ex: prix > 3 fois l'écart interquartile)
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['price'] < (Q1 - 1.5 * IQR)) | (df['price'] > (Q3 + 1.5 * IQR))]

print(f"\nNombre de prix aberrants détectés : {len(outliers)}")

# 5. VISUALISATION DE LA DISTRIBUTION
plt.figure(figsize=(10, 5))
sns.histplot(df['price'], bins=50, kde=True)
plt.title('Distribution des prix (Vérification de la normalité)')
plt.show()